# Running Pangolin on Colab

0. By default, Colab will not use a GPU. To use a GPU, choose: Runtime -> Change runtime type -> GPU.  

1. Install Pangolin and dependencies


In [1]:
!pip install pyvcf gffutils biopython pandas pyfastx
!git clone https://github.com/tkzeng/Pangolin.git
%cd Pangolin
!pip install .
%cd /content

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'pyvcf' when getting requirements to build wheel
Cloning into 'Pangolin'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 210 (delta 33), reused 29 (delta 29), pack-reused 164 (from 1)
Receiving objects: 100% (210/210), 190.05 MiB | 19.88 MiB/s, done.
Resolving deltas: 100% (62/62), done.
Updating files: 100% (79/79), done.
/content/Pangolin
Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pangolin: filename=pangolin-1.

In [2]:
!pip install pyvcf3 gffutils biopython pandas pyfastx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 988.6/988.6 kB 6.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.2/876.2 kB 35.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [gffutils]


2. Upload input files. For details on ways to do this (for example, through Google Drive), see: https://colab.research.google.com/notebooks/io.ipynb


In [5]:
%%bash
set -euo pipefail
mkdir -p pangolin_ref && cd pangolin_ref

UCSC=https://hgdownload.soe.ucsc.edu/goldenPath
GENCODE=https://ftp.ebi.ac.uk/pub/databases/gencode

# 1. Chromosome FASTAs
wget -q -c $UCSC/hg38/chromosomes/chr12.fa.gz
wget -q -c $UCSC/mm39/chromosomes/chr15.fa.gz
gunzip -f chr12.fa.gz chr15.fa.gz
mv chr12.fa hg38_chr12.fa
mv chr15.fa mm39_chr15.fa

# 2. Latest GENCODE GTFs
get_gtf () {
  local dir=$GENCODE/$1/latest_release
  local f=$(curl -s $dir/ | grep -oE 'gencode\.vM?[0-9]+\.annotation\.gtf\.gz' | sort -u)
  echo "Downloading $f"
  wget -q -c -O $2.gtf.gz $dir/$f
}
get_gtf Gencode_human human
get_gtf Gencode_mouse mouse

# 3. Subset to the chromosome of interest
zcat human.gtf.gz | awk '$1=="chr12" || /^#/' > hg38_chr12.gtf
zcat mouse.gtf.gz | awk '$1=="chr15" || /^#/' > mm39_chr15.gtf

total 815M
-rw-r--r-- 1 root root 130M Jan 24  2014 hg38_chr12.fa
-rw-r--r-- 1 root root 238M Nov 12 14:28 hg38_chr12.gtf
-rw-r--r-- 1 root root 119M Jun  9 12:55 human.gtf.gz
-rw-r--r-- 1 root root 102M Sep 10  2020 mm39_chr15.fa
-rw-r--r-- 1 root root 140M Nov 12 14:28 mm39_chr15.gtf
-rw-r--r-- 1 root root  88M Jun  9 12:55 mouse.gtf.gz
drwxr-xr-x 8 root root 4.0K Nov 12 14:28 Pangolin


In [7]:
%%bash
cd pangolin_ref
python Pangolin/scripts/create_db.py --filter None hg38_chr12.gtf
python Pangolin/scripts/create_db.py --filter None mm39_chr15.gtf
ls -lh *.db

Database created: hg38_chr12.db
Database created: mm39_chr15.db
-rw-r--r-- 1 root root 251M Nov 12 14:42 hg38_chr12.db
-rw-r--r-- 1 root root 154M Nov 12 14:43 mm39_chr15.db


3. Run Pangolin

In [3]:
from google.colab import files
files.upload()

Saving pangolin_hs_vs_mm.py to pangolin_hs_vs_mm.py


{'pangolin_hs_vs_mm.py': b'# ============================================================================\n# Pangolin: human vs mouse splice-site strength at the HOXC lncRNA -> HOXC6/HOXC5 junctions\n#\n# Human sites come from hg38 coordinates; mouse sites are located on mm39 chr15 from the\n# mouse ESTs AA553067.1 and BI412975.1 (no mouse annotation used). Any human site without an\n# EST-defined mouse partner (e.g. a HOXC5 acceptor) is mapped to mouse by local alignment.\n#\n# Colab: run on a GPU runtime. Needs pangolin + pyvcf3 installed (see earlier cells).\n# ============================================================================\nimport os, re, gzip, subprocess\nimport importlib.resources as ir\nimport numpy as np, pandas as pd, torch\nfrom pangolin.model import Pangolin, L, W, AR\n\n# ---------------------------------------------------------------- settings\nFLANK = 5000                      # Pangolin context on each side\nHS_FASTA, HS_CHROM = "/content/chr12.fa.gz", "chr1

In [ ]:
# View the last few lines of the output VCF
!tail brca.pangolin.vcf

4. Download the VCF to your local computer

In [ ]:
from google.colab import files
files.download("brca.pangolin.vcf")